# 🌾 Sahrana — Model Training & Exploration
Crop recommendation · Irrigation timing · Resource efficiency

**Algeria Desert Agriculture AI Platform**

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.figsize'] = (10, 5)
matplotlib.rcParams['axes.spines.top'] = False
matplotlib.rcParams['axes.spines.right'] = False

from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report
import joblib

print('Libraries loaded ✅')

## 1. Load Dataset

In [ ]:
df = pd.read_csv('training_data.csv')
print(f'Dataset shape: {df.shape}')
print(f'Columns: {list(df.columns)}')
df.head()

## 2. Data Exploration

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Crop distribution
crop_counts = df['crop'].value_counts()
axes[0].barh(crop_counts.index, crop_counts.values, color='#3a8035', alpha=0.8)
axes[0].set_title('Crop Distribution', fontweight='bold')
axes[0].set_xlabel('Count')

# Water availability
water_counts = df['water_availability'].value_counts()
axes[1].bar(water_counts.index, water_counts.values,
            color=['#c4955a','#3a8035','#2060a0'], alpha=0.8)
axes[1].set_title('Water Availability', fontweight='bold')
axes[1].set_ylabel('Count')

# Temperature distribution
axes[2].hist(df['temperature_c'], bins=20, color='#c05a1f', alpha=0.8, edgecolor='white')
axes[2].set_title('Temperature Distribution (°C)', fontweight='bold')
axes[2].set_xlabel('Temperature (°C)')

plt.tight_layout()
plt.savefig('exploration_plots.png', dpi=120, bbox_inches='tight')
plt.show()
print('Distribution plots saved ✅')

## 3. Encode & Split

In [ ]:
water_encoder = LabelEncoder()
soil_encoder  = LabelEncoder()
df['water_encoded'] = water_encoder.fit_transform(df['water_availability'])
df['soil_encoded']  = soil_encoder.fit_transform(df['soil_type'])

X = df[['temperature_c','peak_solar_hours','water_encoded','soil_encoded','humidity_pct']]

crop_encoder       = LabelEncoder()
irrigation_encoder = LabelEncoder()
efficiency_encoder = LabelEncoder()

y_crop       = crop_encoder.fit_transform(df['crop'])
y_irrigation = irrigation_encoder.fit_transform(df['irrigation_timing'])
y_efficiency = efficiency_encoder.fit_transform(df['resource_efficiency'])

X_train, X_test, yc_train, yc_test, yi_train, yi_test, ye_train, ye_test = train_test_split(
    X, y_crop, y_irrigation, y_efficiency,
    test_size=0.2, random_state=42, stratify=y_crop
)
print(f'Train: {len(X_train)} rows  |  Test: {len(X_test)} rows')
print(f'Crop classes: {list(crop_encoder.classes_)}')

## 4. Train Models

In [ ]:
# Crop → Random Forest (complex, non-linear interactions)
model_crop = RandomForestClassifier(
    n_estimators=500, max_depth=7, random_state=42, class_weight='balanced'
)

# Irrigation → Decision Tree (rule-based timing logic)
model_irrigation = DecisionTreeClassifier(max_depth=5, random_state=42)

# Efficiency → Decision Tree (simple threshold rules)
model_efficiency = DecisionTreeClassifier(max_depth=4, random_state=42)

model_crop.fit(X_train, yc_train)
model_irrigation.fit(X_train, yi_train)
model_efficiency.fit(X_train, ye_train)

print('All models trained ✅')

## 5. Accuracy Results

In [ ]:
crop_acc = accuracy_score(yc_test, model_crop.predict(X_test))
irr_acc  = accuracy_score(yi_test, model_irrigation.predict(X_test))
eff_acc  = accuracy_score(ye_test, model_efficiency.predict(X_test))

# 5-fold cross validation on crop model
cv_scores = cross_val_score(model_crop, X, y_crop, cv=5, scoring='accuracy')

print('=' * 45)
print('MODEL ACCURACY RESULTS')
print('=' * 45)
print(f'Crop recommendation  — test: {crop_acc:.0%}  |  5-fold CV: {cv_scores.mean():.0%} ± {cv_scores.std():.0%}')
print(f'Irrigation timing    — test: {irr_acc:.0%}')
print(f'Resource efficiency  — test: {eff_acc:.0%}')
print()
print('Per-class crop report:')
print(classification_report(
    yc_test, model_crop.predict(X_test),
    target_names=crop_encoder.classes_
))

## 6. Feature Importance

In [ ]:
features = ['temperature_c','peak_solar_hours','water_availability','soil_type','humidity_pct']
importances = model_crop.feature_importances_

fi_df = pd.DataFrame({'feature': features, 'importance': importances})
fi_df = fi_df.sort_values('importance', ascending=True)

plt.figure(figsize=(8, 4))
bars = plt.barh(fi_df['feature'], fi_df['importance'],
                color=['#c4955a' if v < 0.2 else '#3a8035' for v in fi_df['importance']],
                alpha=0.85)
for bar, val in zip(bars, fi_df['importance']):
    plt.text(val + 0.005, bar.get_y() + bar.get_height()/2,
             f'{val:.3f}', va='center', fontsize=10, fontweight='bold')
plt.title('Feature Importance — Crop Recommendation Model', fontweight='bold', pad=12)
plt.xlabel('Importance Score')
plt.tight_layout()
plt.savefig('feature_importance.png', dpi=120, bbox_inches='tight')
plt.show()
print('Feature importance saved ✅')

## 7. Save Model

In [ ]:
joblib.dump({
    'model_crop':          model_crop,
    'model_irrigation':    model_irrigation,
    'model_efficiency':    model_efficiency,
    'water_encoder':       water_encoder,
    'soil_encoder':        soil_encoder,
    'crop_encoder':        crop_encoder,
    'irrigation_encoder':  irrigation_encoder,
    'efficiency_encoder':  efficiency_encoder,
}, 'sahrana_model.pkl')
print('Model saved → sahrana_model.pkl ✅')

## 8. Test Prediction

In [ ]:
def predict_crop(temperature_c, peak_solar_hours, water_availability, soil_type, humidity_pct):
    water_num = water_encoder.transform([water_availability])[0]
    soil_num  = soil_encoder.transform([soil_type])[0]
    input_row = pd.DataFrame(
        [[temperature_c, peak_solar_hours, water_num, soil_num, humidity_pct]],
        columns=['temperature_c','peak_solar_hours','water_encoded','soil_encoded','humidity_pct']
    )
    proba = model_crop.predict_proba(input_row)[0]
    top3  = proba.argsort()[::-1][:3]
    return {
        'recommended_crop':    crop_encoder.inverse_transform([top3[0]])[0],
        'confidence':          f'{proba[top3[0]]:.0%}',
        'top3':                [(crop_encoder.inverse_transform([i])[0], f'{proba[i]:.0%}') for i in top3],
        'irrigation_timing':   irrigation_encoder.inverse_transform(model_irrigation.predict(input_row))[0],
        'resource_efficiency': efficiency_encoder.inverse_transform(model_efficiency.predict(input_row))[0],
    }

test_cases = [
    dict(temperature_c=42, peak_solar_hours=10, water_availability='low',    soil_type='sandy', humidity_pct=15,  label='Classic Saharan desert'),
    dict(temperature_c=28, peak_solar_hours=7,  water_availability='medium', soil_type='loamy', humidity_pct=40,  label='Northern transition zone'),
    dict(temperature_c=32, peak_solar_hours=9,  water_availability='high',   soil_type='clay',  humidity_pct=55,  label='Irrigated oasis'),
]

for tc in test_cases:
    label = tc.pop('label')
    r = predict_crop(**tc)
    print(f'\n{label}:')
    print(f'  Crop:        {r["recommended_crop"]}  ({r["confidence"]} confidence)')
    print(f'  Top 3:       {r["top3"]}')
    print(f'  Irrigation:  {r["irrigation_timing"]}')
    print(f'  Efficiency:  {r["resource_efficiency"]}')